## 1. Data Loading

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))
from src.data_prep import load_billsum, download_cuad_json, load_cuad_raw, flatten_cuad, group_cuad_by_contract
billsum = load_billsum()
print("BillSum sizes:", {k: len(v) for k, v in billsum.items()})

json_path = download_cuad_json('../data/raw/cuad/CUADv1.json')
raw = load_cuad_raw(json_path)
flat = flatten_cuad(raw)
grouped = group_cuad_by_contract(flat)
print("CUAD Total flat rows:", len(flat))
print("CUAD Total distinct contracts:", len(grouped))

BillSum sizes: {'train': 18949, 'test': 3269, 'ca_test': 1237}
CUAD Total flat rows: 20910
CUAD Total distinct contracts: 510


## 2. Preprocessing — Chunking & Tokenization

In [2]:
from src.chunking import chunk_document
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('t5-small')
text = billsum['train'][0]['text']
chunks = chunk_document(text, tokenizer, max_tokens=1024, overlap=100)
print(f"Produced {len(chunks)} chunks for the contract.")

Produced 1 chunks for the contract.


## 3. Extractive Baseline — LexRank

In [3]:
from src.extractive import lexrank_summarize
for i in range(3):
    text = billsum['train'][i]['text']
    summary = lexrank_summarize(text, sentence_count=3)
    print(f"Summary {i+1}: {summary}\n")

Summary 1: (b) Limitation on Liability.-- (1) In general.--Subject to subsection (c), a business entity shall not be subject to civil liability relating to any injury or death occurring at a facility of the business entity in connection with a use of such facility by a nonprofit organization if-- (A) the use occurs outside of the scope of business of the business entity; (B) such injury or death occurs during a period that such facility is used by the nonprofit organization; and (C) the business entity authorized the use of such facility by the nonprofit organization. (2) Application.--This subsection shall apply-- (A) with respect to civil liability under Federal and State law; and (B) regardless of whether a nonprofit organization pays for the use of a facility. (c) Exception for Liability.--Subsection (b) shall not apply to an injury or death that results from an act or omission of a business entity that constitutes gross negligence or intentional misconduct, including any misconduc

Summary 2: (a) In General.--An agency may postpone public disclosure of a human rights record or particular information in a human rights record only if the agency determines that there is clear and convincing evidence that-- (1) the threat to the military defense, intelligence operations, or conduct of foreign relations of the United States raised by public disclosure of the human rights record is of such gravity that it outweighs the public interest, and such public disclosure would reveal-- (A) an intelligence agent whose identity currently requires protection; (B) an intelligence source or method-- (i) which is being utilized, or reasonably expected to be utilized, by the United States Government; (ii) which has not been officially disclosed; and (iii) the disclosure of which would interfere with the conduct of intelligence activities; or (C) any other matter currently relating to the military defense, intelligence operations, or conduct of foreign relations of the United States, t

## 4. Fine-Tuning — DistilBART (Abstractive)\nLoading the saved adapter to save time instead of re-training.

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel, PeftConfig
import torch

peft_model_id = '../outputs/models/distilbart-lora'
try:
    config = PeftConfig.from_pretrained(peft_model_id)
    base_model_name = config.base_model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
    model = PeftModel.from_pretrained(model, peft_model_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    
    text = billsum['train'][500]['text']
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=300)
    print("DistilBART Summary:", tokenizer.decode(out[0], skip_special_tokens=True))
except Exception as e:
    print(f"Model load skipped: {e}")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

DistilBART Summary:  The Higher Education Affordability and Equity Equity Act of 2005 amends the Internal Revenue Code of 1986 (relating to maximum deduction) for interest on education loans to include interest on interest on loans for higher education loans . The Act may be cited as the    (a) Repeal of Dollar Limitation; Increase in Phaseout Beginning Beginning (b) of section 221 of the  Revenue Code (B) of 1986) amends to read as follows: (b), (c) $15,000 ($30,000 in the case of a joint return) $100,000 to $200,000, and (B), $200k ($200k) to each be increased by an amount equal to the taxpayer's modified adjusted gross income for such taxable year, after application of sections 86, 135, 137, and 219, and 469, and 933, respectively . (2) to reduce the amount which bears the same ratio to the amount (but not below zero) under paragraph (2), and (3) the amount of reduction of $15k ($30k) of such deduction under section 221(f) under section 1(f)(1) of the IRS code of 1986 to $20k ($15k)

## 5. Fine-Tuning — T5-Small (Comparison)\nLoading the saved adapter.

In [5]:
peft_model_id = '../outputs/models/t5-lora'
try:
    config = PeftConfig.from_pretrained(peft_model_id)
    base_model_name = config.base_model_name_or_path
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
    model = PeftModel.from_pretrained(model, peft_model_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    
    text = billsum['train'][500]['text']
    inputs = tokenizer("summarize: " + text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_length=300)
    print("T5-Small Summary:", tokenizer.decode(out[0], skip_special_tokens=True))
except Exception as e:
    print(f"Model load skipped: {e}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5-Small Summary: ''Higher Education Affordability and Equity Act of 2005'' is amended to read as follows. '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''


## 6. Clause-Type Tagging (CUAD)

In [6]:
from src.clause_tagger import tag_clauses
test_contract = list(grouped.keys())[0]
tagged = tag_clauses(grouped[test_contract])
print(f"Contract: {test_contract}")
for cat, clauses in tagged.items():
    print(f" - {cat}: {len(clauses)} clauses found")

Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
 - Document Name: 1 clauses found
 - Parties: 5 clauses found
 - Agreement Date: 1 clauses found
 - Effective Date: 2 clauses found
 - Expiration Date: 1 clauses found
 - Renewal Term: 1 clauses found
 - Governing Law: 1 clauses found
 - Exclusivity: 3 clauses found
 - No-Solicit Of Customers: 2 clauses found
 - No-Solicit Of Employees: 1 clauses found
 - Rofr/Rofo/Rofn: 3 clauses found
 - Anti-Assignment: 2 clauses found
 - Price Restrictions: 2 clauses found
 - Minimum Commitment: 5 clauses found
 - License Grant: 2 clauses found
 - Post-Termination Services: 4 clauses found
 - Warranty Duration: 8 clauses found
 - Insurance: 1 clauses found
 - Covenant Not To Sue: 1 clauses found


## 7. Evaluation — ROUGE & Clause Preservation

In [7]:
import pandas as pd
from src.evaluate import compare_models
import json
with open('../outputs/summaries/ten_contract_results.json', 'r') as f:
    res = json.load(f)
df = pd.DataFrame(res)
print(df[['contract_id', 'original_length', 'clause_types_found', 'extractive_clause_types_preserved', 'abstractive_clause_types_preserved']].head())

                                         contract_id  original_length  \
0  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...            54290   
1  WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...            70383   
2  LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...            11475   
3  CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...            15176   
4  ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...            24632   

   clause_types_found  extractive_clause_types_preserved  \
0                  19                                 18   
1                  12                                 11   
2                   6                                  6   
3                  10                                  9   
4                  13                                 12   

   abstractive_clause_types_preserved  
0                                  19  
1                                   4  
2                                   6  
3                                   9  


## 8. Ten-Contract Summarization Run

In [8]:
print("Ten Contract Results loaded from Phase 8:")
print(df.to_string())

Ten Contract Results loaded from Phase 8:
   original_length                                                                                    contract_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             